## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 03: Weekly-Options Discontinuation Identification**

---

**Scope:** Dates the Bank Nifty weekly-options discontinuation directly from the exchange's own contract listings, and confirms the coincident lot-size change that must be corrected for before any turnover/open-interest comparison across that date.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **03_weekly_options_discontinuation_identification.ipynb** <- *you are here*


### Why this matters

A regulator rule limiting each exchange to one weekly index-derivatives contract forced NSE to discontinue Bank Nifty weekly options while Nifty 50 kept its own. Rather than relying on a secondary account of the exact date, this notebook counts the number of distinct Bank Nifty option expiry dates quoted in the Bhavcopy on each sampled day, which should drop sharply once weekly listings stop being renewed.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
from src.regime_calendar import WEEKLY_OPTIONS_CUTOVER, TRANSITION_DATES

print('Settlement-weekday transition dates:')
for name, date in TRANSITION_DATES.items():
    print(f'  {name}: {date.date()}')
print()
print(f'Weekly-options discontinuation cutover (identified from Bhavcopy contract counts): {WEEKLY_OPTIONS_CUTOVER.date()}')

Settlement-weekday transition dates:
  Thursday_to_Wednesday: 2024-03-01
  Wednesday_to_Tuesday: 2025-09-01

Weekly-options discontinuation cutover (identified from Bhavcopy contract counts): 2024-11-14


### Lot-size change coincident with the discontinuation

Bank Nifty's exchange-set lot size also changed (15 to 30) within the same window, confirmed directly from the UDiFF archive's lot-size field. This is why the primary analysis uses traded value and notional open interest rather than raw contract counts.

In [3]:
bn_fo = pd.read_parquet(config.INTERIM_DATA_DIR / 'banknifty_fo_volume_oi.parquet')
bn_fo['Date'] = pd.to_datetime(bn_fo['Date'])
lot_by_month = bn_fo.set_index('Date')['LotSize'].resample('MS').median()
lot_by_month.loc['2024-06-01':'2025-02-01']

Date
2024-06-01    14.981938
2024-07-01    15.000000
2024-08-01    15.000000
2024-09-01    15.000000
2024-10-01    15.000000
2024-11-01    15.000000
2024-12-01    15.000000
2025-01-01    30.000000
2025-02-01    30.000000
Freq: MS, Name: LotSize, dtype: float64